In [15]:
import pandas as pd
import numpy as np
import math

Получение данных о результатах игр Испанской лиги за 30 лет

In [16]:
matches_df = pd.read_csv("LaLiga_Matches.csv")
matches_df

,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR
0,1995-96,02-09-1995,La Coruna,Valencia,3,0,H,2.0,0.0,H
1,1995-96,02-09-1995,Sp Gijon,Albacete,3,0,H,3.0,0.0,H
2,1995-96,03-09-1995,Ath Bilbao,Santander,4,0,H,2.0,0.0,H
3,1995-96,03-09-1995,Ath Madrid,Sociedad,4,1,H,1.0,1.0,D
4,1995-96,03-09-1995,Celta,Compostela,0,1,A,0.0,0.0,D
...,...,...,...,...,...,...,...,...,...,...
11659,2025-26,26-10-2025,Mallorca,Levante,1,1,D,0.0,1.0,A
11660,2025-26,26-10-2025,Real Madrid,Barcelona,2,1,H,2.0,1.0,H
11661,2025-26,26-10-2025,Osasuna,Celta,2,3,A,2.0,1.0,H
11662,2025-26,26-10-2025,Vallecano,Alaves,1,0,H,0.0,0.0,D


Получение данных о результатах в последнем сезоне

In [17]:
last_season_df = matches_df[matches_df["Season"] == "2024-25"].reset_index(drop=True)
last_season_df

,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR
0,2024-25,15-08-2024,Ath Bilbao,Getafe,1,1,D,1.0,0.0,H
1,2024-25,15-08-2024,Betis,Girona,1,1,D,1.0,0.0,H
2,2024-25,16-08-2024,Celta,Alaves,2,1,H,0.0,1.0,A
3,2024-25,16-08-2024,Las Palmas,Sevilla,2,2,D,1.0,1.0,D
4,2024-25,17-08-2024,Osasuna,Leganes,1,1,D,0.0,1.0,A
...,...,...,...,...,...,...,...,...,...,...
375,2024-25,24-05-2025,Getafe,Celta,1,2,A,1.0,1.0,D
376,2024-25,24-05-2025,Vallecano,Mallorca,0,0,D,0.0,0.0,D
377,2024-25,25-05-2025,Girona,Ath Madrid,0,4,A,0.0,0.0,D
378,2024-25,25-05-2025,Villarreal,Sevilla,4,2,H,3.0,1.0,H


Нахождение процентного соотношения ничьих и побед в домашних и выездных матчах

In [18]:
FTHW = (last_season_df["FTR"] == "H").sum()
FTAW = (last_season_df["FTR"] == "A").sum()
FTD = (last_season_df["FTR"] == "D").sum()
total_matches = last_season_df["Date"].count()
last_season_outcome_df = pd.DataFrame({
    "Outcome": ["Home", "Away", "Draw", "Total"],
    "Amount": [FTHW, FTAW, FTD, total_matches],
    "Percentage": [round((FTHW/total_matches)*100, 2), round((FTAW/total_matches)*100, 2), round((FTD/total_matches)*100, 2), 100]
})
last_season_outcome_df


,Outcome,Amount,Percentage
0,Home,169,44.47
1,Away,114,30.00
2,Draw,97,25.53
3,Total,380,100.00


Распределение количества голов за последний сезон (домашние по вертикали, выездные по горизонтали)

In [19]:
goals_distr_np = np.zeros((5, 5))
for idx, row in last_season_df.iterrows():
    FTHG = row['FTHG']
    FTAG = row['FTAG']
    if FTHG >= 4:
        FTHG = 4
    elif FTAG >= 4:
        FTAG = 4
    goals_distr_np[FTHG][FTAG] += 1
goals_distr = pd.DataFrame(goals_distr_np, columns=['0', '1', '2', '3', '4+'], index=['0', '1', '2', '3', '4+'], dtype=int)
goals_distr

,0,1,2,3,4+
0,21,31,13,8,6
1,53,57,37,5,4
2,20,33,16,8,2
3,16,10,9,3,0
4+,6,11,6,5,0


Нахождение средней результативности лиги и домашнего преимущества 

In [20]:
AAG = round(last_season_df['FTAG'].sum()/last_season_df['Date'].count(), 3)
HAG = round(last_season_df['FTHG'].sum()/last_season_df['Date'].count(), 3)
nu = round(np.log(AAG), 3)
nu_home = round(np.log(HAG) - nu, 3)
print(nu)
print(nu_home)

0.154
0.22


Определение средних забитых и пропущенных голов, а также факторов атаки и защиты для каждой команды лиги

In [21]:
teams_dict = {}
for idx, row in last_season_df.iterrows():
    if row['HomeTeam'] in teams_dict:
        teams_dict[row['HomeTeam']][0] += row['FTHG']
        teams_dict[row['HomeTeam']][1] += row['FTAG']
        teams_dict[row['HomeTeam']][2] += 1
    else:
        teams_dict[row['HomeTeam']] = [row['FTHG'], row['FTAG'], 1]
    
    if row['AwayTeam'] in teams_dict:
        teams_dict[row['AwayTeam']][0] += row['FTAG']
        teams_dict[row['AwayTeam']][1] += row['FTHG']
        teams_dict[row['AwayTeam']][2] += 1
    else:
        teams_dict[row['AwayTeam']] = [row['FTAG'], row['FTHG'], 1]
teams_df = pd.DataFrame.from_dict(teams_dict, orient='index')

for i in teams_df.index:
    scored = pd.to_numeric(teams_df.loc[i, 0])
    missed = pd.to_numeric(teams_df.loc[i, 1])
    total = pd.to_numeric(teams_df.loc[i, 2])
    teams_df.loc[i, 0] = round(scored / total, 3)
    teams_df.loc[i, 1] = round(missed / total, 3)
teams_df = teams_df.drop(2, axis=1)
teams_df = teams_df.rename(columns={0:'Avg goals scored', 1:'Avg goals missed'})
teams_df['Atk factor'] = round(np.log(teams_df['Avg goals scored']) - nu, 3)
teams_df['Def factor'] = round(np.log(teams_df['Avg goals missed']) - nu, 3)
teams_df

/tmp/ipykernel_4209/2684057240.py:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '1.421' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  teams_df.loc[i, 0] = round(scored / total, 3)
/tmp/ipykernel_4209/2684057240.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.763' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  teams_df.loc[i, 1] = round(missed / total, 3)


,Avg goals scored,Avg goals missed,Atk factor,Def factor
Ath Bilbao,1.421,0.763,0.197,-0.424
Getafe,0.895,1.026,-0.265,-0.128
Betis,1.500,1.316,0.251,0.121
Girona,1.158,1.579,-0.007,0.303
Celta,1.553,1.500,0.286,0.251
Alaves,1.000,1.263,-0.154,0.079
Las Palmas,1.053,1.605,-0.102,0.319
Sevilla,1.105,1.447,-0.054,0.215
Osasuna,1.263,1.368,0.079,0.159
Leganes,1.026,1.474,-0.128,0.234


Рассмотрим матч "Барселона" - "Реал Мадрид" 11.05.2025

In [28]:
test_match = last_season_df[last_season_df.index == 347]
test_match

,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR
347,2024-25,11-05-2025,Barcelona,Real Madrid,4,3,H,4.0,2.0,H


In [ ]:
Home_est = np.exp(nu + nu_home + teams_df['Atk factor'][test_match['HomeTeam']].values + teams_df['Def factor'][test_match['AwayTeam']].values).item()
Away_est = np.exp(nu + teams_df['Atk factor'][test_match['AwayTeam']].values + teams_df['Def factor'][test_match['HomeTeam']].values).item()
print(f"Ожидаемое количество голов, забитое домашней командой: {round(Home_est, 3)}")
print(f"Ожидаемое количество голов, забитое гостевой командой: {round(Away_est, 3)}")

Ожидаемое количество голов, забитое домашней командой: 1.181
Ожидаемое количество голов, забитое гостевой командой: 2.545


In [25]:
def PoisonDistr(k, l):
    return (l **k * np.e**(-l))/(math.factorial(k))

In [ ]:
row = 7
col = 10
match_goals_distr = np.zeros((row, col))
Home_W_prob = 0
Away_W_prob = 0
Draw_prob = 0
for i in range(row):
    for j in range(col):
        res = PoisonDistr(i, Away_est) * PoisonDistr(j, Home_est)
        match_goals_distr[i][j] = round(res, 3)
        if i > j:
            Away_W_prob += res
        elif j > i:
            Home_W_prob += res
        elif i == j:
            Draw_prob += res
match_goals_distr_df = pd.DataFrame(match_goals_distr)
print(f"Вероятность победы {str(test_match["HomeTeam"].values)[2:-2]}: {round(Home_W_prob, 2)}")
print(f"Вероятность победы {str(test_match["AwayTeam"].values)[2:-2]}: {round(Away_W_prob, 2)}")
print(f"Вероятность ничьи: {round(Draw_prob, 2)}")
print(f"Наиболее вероятный счет в матче: {match_goals_distr_df.max().idxmax()}:{match_goals_distr_df.idxmax()[0]}")
print(f"Вероятность появления данного счета: {match_goals_distr_df.max().max()}")
match_goals_distr_df

Вероятность победы Sevilla: 0.16
Вероятность победы Real Madrid: 0.65
Вероятность ничьи: 0.17
Наиболее вероятный счет в матче: 1:2
Вероятность появления данного счета: 0.092


,0,1,2,3,4,5,6,7,8,9
0,0.024,0.028,0.017,0.007,0.002,0.000,0.0,0.0,0.0,0.0
1,0.061,0.072,0.043,0.017,0.005,0.001,0.0,0.0,0.0,0.0
2,0.078,0.092,0.054,0.021,0.006,0.001,0.0,0.0,0.0,0.0
3,0.066,0.078,0.046,0.018,0.005,0.001,0.0,0.0,0.0,0.0
4,0.042,0.050,0.029,0.012,0.003,0.001,0.0,0.0,0.0,0.0
5,0.021,0.025,0.015,0.006,0.002,0.000,0.0,0.0,0.0,0.0
6,0.009,0.011,0.006,0.002,0.001,0.000,0.0,0.0,0.0,0.0


Нахождение "**чистых**" коэффициентов для букмекеров

In [ ]:
print(f"Коэффициент на победу {str(test_match["HomeTeam"].values)[2:-2]}: {round(1/Home_W_prob, 2)}")
print(f"Коэффициент на победу {str(test_match["AwayTeam"].values)[2:-2]}: {round(1/Away_W_prob, 2)}")
print(f"Коэффициент на ничью: {round(1/Draw_prob, 2)}")

Коэффициент на победу Sevilla: 6.33
Коэффициент на победу Real Madrid: 1.53
Коэффициент на ничью: 5.78
